# 实验 24 — dbt 多 target 与环境感知

**目标**：用 `--target` 切环境，在模型里用 `{{ target.name }}` 写环境感知逻辑（dev 采样 / prod 全量等）。

本仓库的 [profiles.yml](../dbt_project/profiles.yml) 已经多加了一个 `prod` target，它指到 `main_prod` schema（同一个 duckdb 文件，仅 schema 隔离，方便本地学习）。

## 常见生产模式

- `target.name` 在 model 里分支：dev 只跑近 30 天，prod 跑全量
- `generate_schema_name` macro 重写 schema：dev 加用户前缀（`dbt_alice_marts`），prod 用纯 `marts`
- `target.name` 决定 alerts：on-run-end 仅 prod 发钉钉/Slack

In [ ]:
import subprocess, os, duckdb
def dbt(*a):
    r = subprocess.run(['uv','run','dbt',*a], capture_output=True, text=True,
                       cwd='../dbt_project',
                       env={**os.environ,'DBT_PROFILES_DIR':'.'})
    return r.stdout + r.stderr

print('=== 默认 dev target ===')
print(dbt('debug')[-800:])

In [ ]:
print('=== 切到 prod target ===')
print(dbt('debug','--target','prod')[-800:])

In [ ]:
# 跑 prod 实际会在 main_prod schema 建表
print(dbt('build','--target','prod','--exclude','snp_currencies')[-1500:])

In [ ]:
con = duckdb.connect('../data/sandbox.duckdb')
print('dev tables:')
print(con.execute("select table_name from information_schema.tables where table_schema='main' order by 1").df())
print()
print('prod tables:')
print(con.execute("select table_name from information_schema.tables where table_schema='main_prod' order by 1").df())

In [ ]:
# 演示 model 里用 target.name 分支：dev 限 30 天，prod 全量
from pathlib import Path
m = Path('../dbt_project/models/marts/fct_daily_rates.sql')
orig = m.read_text()
m.write_text(orig.replace(
    'from stg\njoin dim on stg.currency = dim.currency_code',
    '''from stg
join dim on stg.currency = dim.currency_code
{% if target.name == 'dev' %}
  where stg.rate_date > current_date - interval '30 day'
{% endif %}'''
))
print('compile dev:')
print(dbt('compile','--select','fct_daily_rates')[-200:])
p = Path('../dbt_project/target/compiled/dlt_dbt_sandbox/models/marts/fct_daily_rates.sql')
print(p.read_text())

In [ ]:
print('compile prod:')
print(dbt('compile','--select','fct_daily_rates','--target','prod')[-200:])
print(p.read_text())

In [ ]:
# 还原
m.write_text(orig)

## 生产环境踩坑提示

- **target.name 不是 magic**：它就是 profiles.yml 里的 key。常见命名 `dev` / `ci` / `prod`。CI 用专属 target，避免 “dev” 在 CI 跑出意料外的行为。
- **`generate_schema_name` 自定义** 是 dbt 项目里最常 override 的 macro。模板：
  ```sql
  {% macro generate_schema_name(custom_schema_name, node) -%}
      {%- if target.name == 'prod' or custom_schema_name is none -%}
          {{ target.schema }}
      {%- else -%}
          {{ target.schema }}_{{ custom_schema_name | trim }}
      {%- endif -%}
  {%- endmacro %}
  ```
- **dev 用户隔离**：让每个分析师 dev target 用自己的 schema（`{{ env_var('USER') }}_dev`），避免互相覆盖。
- **prod 凭证保护**：profiles.yml 不要把 prod password 写明文；用 `password: "{{ env_var('DBT_PROD_PASSWORD') }}"`。
- **CI target 通常和 prod 同库**，但写到 `dbt_ci_<run_id>` schema，run 完即销毁。

## 思考题

- 自己实现一个 `generate_schema_name` 让 dev 自动加用户前缀。
- 在你公司，prod 和 dev 是同库不同 schema，还是不同的 warehouse / project？各自的 pros/cons？
- `dbt seed` 跑在 prod target 上时，seed CSV 会去到 prod schema —— 你想这么做吗？如果不想，怎么阻止？